In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev_pnl",
    choices=["fq_dev_pnl", "fq_test_pnl", "fq_prod_pnl"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="other",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="gl_report",
    choices=["discount", "sales", "cost", "gl_report", "pnl_budget_flat_data", "dim_coa_master"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `fq_dev_extloc_staging`"
).select("url").collect()[0][0]

checkpoint = 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/checkpoints/'

In [0]:
print(staging)

In [0]:
from pyspark.sql.functions import *

cdc_raw_data = spark.read.option('multiline', False).format('json').load(f'{staging}/FoodQuest/Netsuite/GL_Report/ALBAIK/2026/JAN/gl_report.json').limit(1)
display(cdc_raw_data)

In [0]:
%sql CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_pnl_catalog.bronze.gl_report 
USING DELTA
LOCATION 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/bronze/gl_report' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
import time
from pyspark.sql.functions import col, expr, current_timestamp, regexp_replace, lit, substring, substring_index, to_timestamp, size

bronzeDF = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "json")
                .option('multiline', False)
                .option("cloudFiles.allowOverwrites", "true") #re-ingest files if they are overwritten or update
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.schemaLocation", f'{checkpoint}/{source}/{domain}/streaming/schema_gl_report')
                .load(f'{staging}/FoodQuest/Netsuite/GL_Report/ALBAIK/*/gl_report.json')
            )
# display(bronzeDF)

(bronzeDF.withColumn("ingestion_ts", current_timestamp())
        .withColumn('file_name', substring_index(col('_metadata.file_name'), '.', 1))
        .withColumn('file_path', regexp_replace(col('_metadata.file_path'), '%20', ' '))
        .withColumn('sys_id', expr('uuid()'))
        .writeStream
        .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_gl_report')
        .trigger(availableNow=True)
        .queryName(f'{domain}WriteStream')
        .outputMode('append')
        .toTable(f"`{environment}_catalog`.`bronze`.`{domain}`", mergeSchema=True)
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows,
  count(DISTINCT file_path) as total_files
FROM fq_dev_pnl_catalog.bronze.gl_report;

In [0]:
%sql select * from fq_dev_pnl_catalog.bronze.gl_report

In [0]:
for query in spark.streams.active:
    query.stop()